# Step 3 - Final Modelling Dataset

This notebook builds the final supervised-learning dataset for the neural network.

The goal is to predict the one-year-ahead SPF forecast error:

```text
forecast_error = realized_actual_value - SPF_rolling_1y_forecast
```

The dataset combines three blocks of information:

- cleaned ECB SPF forecasts from Step 1
- maintained euro-area macro and financial predictors from EA-MD-QD
- leak-safe historical forecast-error features

I keep the target construction, date alignment, and leakage checks explicit because
those choices matter more than the neural-network architecture for whether the
experiment is credible.

## Imports and Paths

The final predictor block now comes from the maintained EA-MD-QD workbook that we
already processed with the package's own transformation and imputation routine.
The workbook is quarterly, so it can be aligned naturally to SPF survey rounds.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("Data")

SPF_WIDE_PATH = DATA_DIR / "spf_clean_wide.csv"
EA_MD_QD_PATH = DATA_DIR / "EA-MD-QD-2026-04" / "data_TR2" / "EAdataQM_TR2.xlsx"
FINAL_DATASET_PATH = DATA_DIR / "final_model_dataset.csv"

# Use the latest fully completed EA-MD-QD quarter before the SPF survey quarter.
# This conservative one-quarter lag avoids giving the model same-quarter
# information that may not have been public when respondents answered the survey.
EA_PREDICTOR_LAG_QUARTERS = 1

print("SPF input:", SPF_WIDE_PATH)
print("EA-MD-QD input:", EA_MD_QD_PATH)
print("Final output:", FINAL_DATASET_PATH)

## Load SPF and EA-MD-QD Inputs

`spf_clean_wide.csv` is one row per forecaster, survey round, and variable.
`EAdataQM_TR2.xlsx` is the processed quarterly EA-MD-QD panel. EA here means
**euro area**, which is the correct geographic aggregate for the ECB SPF HICP
and real GDP questions.

In [ ]:
spf = pd.read_csv(SPF_WIDE_PATH)
ea_md_qd = pd.read_excel(EA_MD_QD_PATH)

print("SPF wide shape:", spf.shape)
print("EA-MD-QD shape:", ea_md_qd.shape)

display(spf.head())
display(ea_md_qd.head())

## Clean Target Labels and Build Survey Dates

Some target-period columns are stored as numeric-looking strings after CSV
export, for example `2000.0`. I convert them back to clean labels before any
merge. I also create a real quarterly date for each survey round.

The date rule is:

- `survey_quarter_start`: the first day of the SPF survey quarter
- `ea_reference_date`: the EA-MD-QD quarter allowed as information

With the default one-quarter lag, a 2012Q1 SPF row uses 2011Q4 EA-MD-QD
predictors. This is intentionally conservative.

In [ ]:
def clean_target_label(value):
    """Convert target-period labels like 2000.0 back to '2000'."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


target_label_cols = [
    "rolling_1y_target",
    "rolling_2y_target",
    "rolling_longer_target",
    "next_year_target",
    "current_year_target",
]

for col in target_label_cols:
    if col in spf.columns:
        spf[col] = spf[col].map(clean_target_label)

survey_period = pd.PeriodIndex(
    spf["survey_year"].astype(str) + "Q" + spf["survey_quarter"].astype(str),
    freq="Q",
)

spf["survey_quarter_start"] = survey_period.to_timestamp(how="start")
spf["survey_quarter_end"] = survey_period.to_timestamp(how="end").normalize()
spf["ea_reference_date"] = (
    survey_period - EA_PREDICTOR_LAG_QUARTERS
).to_timestamp(how="start")

display(
    spf[
        [
            "survey_round",
            "survey_year",
            "survey_quarter",
            "survey_quarter_start",
            "ea_reference_date",
            "variable",
            "rolling_1y_target",
        ]
    ].head(10)
)

## Parse Rolling One-Year Target Dates

The target date is useful for constructing past-error features. A previous SPF
error is only usable once its target period has occurred.

- HICP rolling targets are monthly, for example `2000Dec`
- RGDP rolling targets are quarterly, for example `2000Q4`

In [ ]:
MONTH_ABBR_TO_NUM = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}


def parse_spf_target_date(variable, target_label):
    """Return the end date of the SPF target period."""
    if pd.isna(target_label):
        return pd.NaT

    text = str(target_label)

    if variable == "HICP":
        year = int(text[:4])
        month_text = text[4:]
        month = MONTH_ABBR_TO_NUM[month_text]
        return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)

    if variable == "RGDP":
        return pd.Period(text, freq="Q").to_timestamp(how="end").normalize()

    return pd.NaT


spf["target_date"] = [
    parse_spf_target_date(variable, target)
    for variable, target in zip(spf["variable"], spf["rolling_1y_target"])
]

display(
    spf[
        [
            "survey_round",
            "variable",
            "rolling_1y_target",
            "target_date",
            "survey_quarter_start",
        ]
    ].drop_duplicates().head(12)
)

## Keep Rows with One-Year SPF Forecasts

Step 1 intentionally uses outer merges across forecast horizons. That is useful
for an archive table, but the neural network's target is specifically the
rolling one-year-ahead error. Therefore the final training dataset must keep only
rows where `rolling_1y_forecast` is available.

In [ ]:
rows_before = len(spf)
model = spf.loc[spf["rolling_1y_forecast"].notna()].copy()
rows_after = len(model)

print("Rows before filtering:", rows_before)
print("Rows after requiring rolling_1y_forecast:", rows_after)
print("Rows removed:", rows_before - rows_after)

assert model["rolling_1y_forecast"].notna().all()
display(model.groupby("variable").size().rename("n_rows").reset_index())

## Add SPF Consensus Features

The raw individual forecast is not the only useful SPF signal. The model may also
learn from how far a forecaster is from the survey consensus and how dispersed
forecasts are in that round.

In [ ]:
consensus = (
    model
    .groupby(["survey_round", "variable"], as_index=False)
    .agg(
        consensus_mean_rolling_1y=("rolling_1y_forecast", "mean"),
        consensus_median_rolling_1y=("rolling_1y_forecast", "median"),
        forecast_disagreement_rolling_1y=("rolling_1y_forecast", "std"),
        n_forecasters_rolling_1y=("rolling_1y_forecast", "count"),
    )
)

model = model.merge(
    consensus,
    on=["survey_round", "variable"],
    how="left",
    validate="many_to_one",
)

model["forecast_disagreement_rolling_1y"] = (
    model["forecast_disagreement_rolling_1y"].fillna(0.0)
)
model["distance_from_consensus_rolling_1y"] = (
    model["rolling_1y_forecast"] - model["consensus_mean_rolling_1y"]
)
model["abs_distance_from_consensus_rolling_1y"] = (
    model["distance_from_consensus_rolling_1y"].abs()
)

display(
    model[
        [
            "survey_round",
            "variable",
            "rolling_1y_forecast",
            "consensus_mean_rolling_1y",
            "forecast_disagreement_rolling_1y",
            "distance_from_consensus_rolling_1y",
        ]
    ].head()
)

## Add Basic Time and Variable Features

`variable_RGDP` lets a pooled model distinguish inflation rows from real GDP rows.
The NN notebook can still train separate HICP and RGDP models, which is the main
specification we currently prefer.

In [ ]:
model["survey_round_index"] = (
    (model["survey_year"] - model["survey_year"].min()) * 4
    + (model["survey_quarter"] - 1)
)
model["variable_RGDP"] = (model["variable"] == "RGDP").astype(int)

display(
    model[
        [
            "survey_round",
            "survey_year",
            "survey_quarter",
            "survey_round_index",
            "variable",
            "variable_RGDP",
        ]
    ].drop_duplicates().head(10)
)

## Download Realized Outcomes from Eurostat

The target variable must use the realized value matching the SPF question. I use
Eurostat for the realized outcomes because it gives the official HICP annual
rate of change and real GDP year-on-year growth series.

EA-MD-QD is used for predictors. Its transformed GDP and HICP columns are useful
signals available to the model, but the supervised-learning label should be built
from the same official realized concepts as the SPF forecast targets.

In [ ]:
EUROSTAT_BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"


def decode_jsonstat_position(position, sizes):
    """Decode a flat JSON-stat value position into dimension coordinates."""
    coords = []
    remaining = int(position)
    for size in reversed(sizes):
        coords.append(remaining % size)
        remaining //= size
    return list(reversed(coords))


def fetch_eurostat_series(dataset, params, value_name):
    """Fetch one Eurostat JSON-stat series and return a period-indexed Series."""
    url = f"{EUROSTAT_BASE_URL}/{dataset}"
    request_params = {"format": "JSON", "lang": "en", **params}
    response = requests.get(url, params=request_params, timeout=90)
    response.raise_for_status()
    payload = response.json()

    if "value" not in payload:
        return pd.Series(dtype="float64", name=value_name)

    dim_ids = payload["id"]
    sizes = payload["size"]
    category_labels = {}
    for dim in dim_ids:
        index_map = payload["dimension"][dim]["category"]["index"]
        category_labels[dim] = {pos: label for label, pos in index_map.items()}

    values = payload["value"]
    if isinstance(values, list):
        items = [(i, value) for i, value in enumerate(values)]
    else:
        items = [(int(i), value) for i, value in values.items()]

    records = []
    for flat_position, value in items:
        if value is None:
            continue
        coords = decode_jsonstat_position(flat_position, sizes)
        labels = {
            dim: category_labels[dim][coord]
            for dim, coord in zip(dim_ids, coords)
        }
        records.append({"period": labels["time"], value_name: float(value)})

    if not records:
        return pd.Series(dtype="float64", name=value_name)

    out = (
        pd.DataFrame(records)
        .drop_duplicates("period")
        .set_index("period")
        .sort_index()[value_name]
    )
    out.name = value_name
    return out


MONTH_ABBR = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}


def eurostat_month_to_spf_target(period):
    """Convert Eurostat monthly periods like '2000-12' to SPF labels like '2000Dec'."""
    if "-M" in period:
        year, month = period.split("-M")
    else:
        year, month = period.split("-")
    return f"{year}{MONTH_ABBR[int(month)]}"


def eurostat_quarter_to_spf_target(period):
    """Convert Eurostat quarterly periods like '2000-Q4' to SPF labels like '2000Q4'."""
    return period.replace("-Q", "Q").replace("-", "")

In [ ]:
# HICP: annual rate of change, all-items HICP.
hicp_common_params = {
    "freq": "M",
    "unit": "RCH_A",
    "coicop": "CP00",
    "sinceTimePeriod": "1998-01",
}

hicp_ea20 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA20"},
    "hicp_ea20",
)
hicp_ea19 = fetch_eurostat_series(
    "prc_hicp_manr",
    {**hicp_common_params, "geo": "EA19"},
    "hicp_ea19",
)

hicp_combined = pd.concat([hicp_ea20, hicp_ea19], axis=1)
hicp_combined["actual_value"] = hicp_combined["hicp_ea20"].combine_first(
    hicp_combined["hicp_ea19"]
)
hicp_combined["actual_source"] = np.where(
    hicp_combined["hicp_ea20"].notna(),
    "Eurostat prc_hicp_manr EA20",
    "Eurostat prc_hicp_manr EA19 fallback",
)
hicp_actuals = (
    hicp_combined
    .loc[hicp_combined["actual_value"].notna(), ["actual_value", "actual_source"]]
    .reset_index()
    .rename(columns={"index": "eurostat_period", "period": "eurostat_period"})
)
hicp_actuals["variable"] = "HICP"
hicp_actuals["rolling_1y_target"] = hicp_actuals["eurostat_period"].map(
    eurostat_month_to_spf_target
)

# Real GDP growth: chain-linked volumes, percentage change from the same
# quarter of the previous year. This matches a year-on-year growth concept.
gdp_yoy = fetch_eurostat_series(
    "namq_10_gdp",
    {
        "freq": "Q",
        "unit": "CLV_PCH_SM",
        "s_adj": "SCA",
        "na_item": "B1GQ",
        "geo": "EA20",
        "sinceTimePeriod": "1998-Q1",
    },
    "actual_value",
)

gdp_actuals = gdp_yoy.reset_index().rename(
    columns={"index": "eurostat_period", "period": "eurostat_period"}
)
gdp_actuals["variable"] = "RGDP"
gdp_actuals["rolling_1y_target"] = gdp_actuals["eurostat_period"].map(
    eurostat_quarter_to_spf_target
)
gdp_actuals["actual_source"] = "Eurostat namq_10_gdp EA20"

actuals = pd.concat(
    [
        hicp_actuals[
            ["variable", "rolling_1y_target", "actual_value", "actual_source", "eurostat_period"]
        ],
        gdp_actuals[
            ["variable", "rolling_1y_target", "actual_value", "actual_source", "eurostat_period"]
        ],
    ],
    ignore_index=True,
)

actuals = actuals.drop_duplicates(["variable", "rolling_1y_target"])

print("HICP Eurostat periods:", hicp_actuals["eurostat_period"].min(), "to", hicp_actuals["eurostat_period"].max())
print("RGDP Eurostat periods:", gdp_actuals["eurostat_period"].min(), "to", gdp_actuals["eurostat_period"].max())
print("Actuals shape:", actuals.shape)
display(actuals.head())
display(actuals.tail())

## Create the Neural-Network Target Variables

The main label is `forecast_error`.

```text
forecast_error = actual_value - rolling_1y_forecast
```

This sign convention means a positive NN prediction says the raw SPF forecast
should be revised upward, and a negative prediction says it should be revised
downward.

In [ ]:
model = model.merge(
    actuals,
    on=["variable", "rolling_1y_target"],
    how="left",
    validate="many_to_one",
)

model["forecast_error"] = model["actual_value"] - model["rolling_1y_forecast"]
model["abs_forecast_error"] = model["forecast_error"].abs()
model["squared_forecast_error"] = model["forecast_error"] ** 2

missing_actuals = model.loc[model["actual_value"].isna()].copy()

print("Rows after actual merge:", len(model))
print("Rows without realized actual value yet:", len(missing_actuals))
if len(missing_actuals) > 0:
    display(
        missing_actuals
        .groupby(["variable", "rolling_1y_target"], as_index=False)
        .size()
        .sort_values(["variable", "rolling_1y_target"])
        .head(15)
    )

display(
    model[
        [
            "survey_round",
            "variable",
            "fct_source",
            "rolling_1y_target",
            "rolling_1y_forecast",
            "actual_value",
            "forecast_error",
            "actual_source",
        ]
    ].head()
)

## Add Leak-Safe Past Forecast-Error Features

Yes, past prediction errors are a sensible feature. They capture persistent
forecaster bias, for example a forecaster who has repeatedly underpredicted
inflation.

The important rule is that the model must not see future errors. For each row,
I only use errors whose target period ended at least one full quarter before
the current SPF survey quarter. This publication buffer is conservative, but
it avoids relying on outcomes that may not have been released yet. I add two versions:

- forecaster-variable history, such as forecaster 12's previous HICP errors
- variable-level history, such as all previous HICP errors

If no history exists yet, the mean features are set to zero and the count feature
is zero. The count tells the neural network whether the historical mean is based
on real evidence or just the no-history fallback.

In [ ]:
def merge_past_error_history(frame, keys, prefix):
    """Merge cumulative past-error summaries without using future target values."""
    out = frame.copy()

    history = out.loc[
        out["forecast_error"].notna() & out["target_date"].notna(),
        [*keys, "target_date", "forecast_error", "abs_forecast_error"],
    ].copy()

    history = (
        history
        .groupby([*keys, "target_date"], as_index=False)
        .agg(
            error_sum=("forecast_error", "sum"),
            abs_error_sum=("abs_forecast_error", "sum"),
            error_count=("forecast_error", "size"),
        )
        .sort_values([*keys, "target_date"])
    )

    history[f"{prefix}_cum_error_sum"] = history.groupby(keys)["error_sum"].cumsum()
    history[f"{prefix}_cum_abs_error_sum"] = history.groupby(keys)["abs_error_sum"].cumsum()
    history[f"{prefix}_past_error_count"] = history.groupby(keys)["error_count"].cumsum()

    history = history.rename(columns={"target_date": "known_target_date"})
    history = history[
        [
            *keys,
            "known_target_date",
            f"{prefix}_cum_error_sum",
            f"{prefix}_cum_abs_error_sum",
            f"{prefix}_past_error_count",
        ]
    ]

    left = (
        out
        .reset_index()
        .sort_values(["past_error_reference_date", *keys])
    )
    right = history.sort_values(["known_target_date", *keys])

    merged = pd.merge_asof(
        left,
        right,
        by=keys,
        left_on="past_error_reference_date",
        right_on="known_target_date",
        direction="backward",
    )

    merged = merged.sort_values("index").set_index("index")

    count_col = f"{prefix}_past_error_count"
    mean_col = f"{prefix}_past_error_mean"
    abs_mean_col = f"{prefix}_past_abs_error_mean"

    counts = merged[count_col].fillna(0.0)
    out[count_col] = counts.astype(int).to_numpy()

    out[mean_col] = np.where(
        counts > 0,
        merged[f"{prefix}_cum_error_sum"] / counts,
        0.0,
    )
    out[abs_mean_col] = np.where(
        counts > 0,
        merged[f"{prefix}_cum_abs_error_sum"] / counts,
        0.0,
    )

    return out, [mean_col, abs_mean_col, count_col]


# A target is treated as usable history only after a full-quarter publication buffer.
# Example: for a 2012Q1 survey, the latest allowed target period is 2011Q3.
model_survey_period = pd.PeriodIndex(
    model["survey_year"].astype(str) + "Q" + model["survey_quarter"].astype(str),
    freq="Q",
)
model["past_error_reference_date"] = (
    model_survey_period - 2
).to_timestamp(how="end").normalize()

model, forecaster_error_cols = merge_past_error_history(
    model,
    keys=["fct_source", "variable"],
    prefix="forecaster_variable",
)
model, variable_error_cols = merge_past_error_history(
    model,
    keys=["variable"],
    prefix="variable",
)

past_error_feature_cols = forecaster_error_cols + variable_error_cols

print("Past-error feature columns:", past_error_feature_cols)
display(
    model[
        [
            "survey_round",
            "variable",
            "fct_source",
            "rolling_1y_target",
            "target_date",
            "past_error_reference_date",
            *past_error_feature_cols,
        ]
    ].head(12)
)

## Align EA-MD-QD Predictors to SPF Survey Rounds

The EA-MD-QD panel is quarterly. I merge it to SPF rows by survey round using
`merge_asof`, which means each survey round receives the most recent EA-MD-QD
observation at or before `ea_reference_date`.

This produces a many-to-one merge: many individual SPF forecasts share the same
euro-area macro information for the survey round.

In [ ]:
ea_md_qd = ea_md_qd.rename(columns={"Date": "ea_md_qd_date"}).copy()
ea_md_qd["ea_md_qd_date"] = pd.to_datetime(ea_md_qd["ea_md_qd_date"])

ea_feature_cols = [col for col in ea_md_qd.columns if col != "ea_md_qd_date"]
ea_md_qd[ea_feature_cols] = ea_md_qd[ea_feature_cols].apply(pd.to_numeric, errors="coerce")

assert not np.isinf(ea_md_qd[ea_feature_cols].to_numpy()).any()

survey_ea_keys = (
    model[
        [
            "survey_round",
            "survey_year",
            "survey_quarter",
            "survey_quarter_start",
            "ea_reference_date",
        ]
    ]
    .drop_duplicates("survey_round")
    .sort_values("ea_reference_date")
)

survey_ea = pd.merge_asof(
    survey_ea_keys,
    ea_md_qd.sort_values("ea_md_qd_date"),
    left_on="ea_reference_date",
    right_on="ea_md_qd_date",
    direction="backward",
)

rows_before_ea = len(model)
model = model.merge(
    survey_ea[["survey_round", "ea_md_qd_date", *ea_feature_cols]],
    on="survey_round",
    how="left",
    validate="many_to_one",
)
assert len(model) == rows_before_ea

print("EA-MD-QD predictor columns:", len(ea_feature_cols))
print("EA-MD-QD date range:", ea_md_qd["ea_md_qd_date"].min(), "to", ea_md_qd["ea_md_qd_date"].max())
print("Rows with no EA-MD-QD predictor match:", model[ea_feature_cols].isna().all(axis=1).sum())

display(
    survey_ea[
        [
            "survey_round",
            "survey_quarter_start",
            "ea_reference_date",
            "ea_md_qd_date",
            "GDP_EA",
            "HICPOV_EA",
            "UNETOT_EA",
        ]
    ].head(12)
)

## Keep Only Rows That Can Train the Model

A supervised row needs:

- a one-year-ahead SPF forecast
- the realized actual value for the target period
- the constructed forecast-error label
- a successful EA-MD-QD macro predictor match

Recent SPF rows whose target has not yet been realized are correctly excluded
from the training file. Rows before the EA-MD-QD sample starts are also
excluded, because otherwise the NN would learn from rows where the full macro
block is missing.

In [ ]:
has_actual_value = model["actual_value"].notna()
has_macro_predictors = model[ea_feature_cols].notna().any(axis=1)

trainable = model.loc[has_actual_value & has_macro_predictors].copy()

print("Rows available for supervised training:", len(trainable))
print("Rows excluded because the actual value is not available yet:", (~has_actual_value).sum())
print("Rows excluded because EA-MD-QD macro predictors are missing:", (has_actual_value & ~has_macro_predictors).sum())

assert trainable["rolling_1y_forecast"].notna().all()
assert trainable["actual_value"].notna().all()
assert trainable["forecast_error"].notna().all()
assert trainable[ea_feature_cols].notna().any(axis=1).all()

display(trainable.groupby("variable").size().rename("n_rows").reset_index())

## Define Column Groups

I separate identifiers, target columns, and candidate feature columns.

`actual_value`, `forecast_error`, `abs_forecast_error`, and
`squared_forecast_error` are outcomes, not model inputs. The main NN label is
`forecast_error`.

In [ ]:
identifier_cols = [
    "survey_round",
    "survey_year",
    "survey_quarter",
    "survey_quarter_start",
    "survey_quarter_end",
    "survey_round_index",
    "ea_reference_date",
    "ea_md_qd_date",
    "variable",
    "variable_RGDP",
    "fct_source",
    "rolling_1y_target",
    "target_date",
    "past_error_reference_date",
    "eurostat_period",
    "actual_source",
]

target_cols = [
    "actual_value",
    "forecast_error",
    "abs_forecast_error",
    "squared_forecast_error",
]

spf_feature_cols = [
    "rolling_1y_forecast",
    "rolling_2y_forecast",
    "rolling_longer_forecast",
    "next_year_forecast",
    "current_year_forecast",
    "consensus_mean_rolling_1y",
    "consensus_median_rolling_1y",
    "forecast_disagreement_rolling_1y",
    "n_forecasters_rolling_1y",
    "distance_from_consensus_rolling_1y",
    "abs_distance_from_consensus_rolling_1y",
]

candidate_feature_cols = [
    "survey_year",
    "survey_quarter",
    "survey_round_index",
    "variable_RGDP",
    *spf_feature_cols,
    *past_error_feature_cols,
    *ea_feature_cols,
]

final_cols = (
    identifier_cols
    + target_cols
    + spf_feature_cols
    + past_error_feature_cols
    + ea_feature_cols
)

final_cols = [col for col in final_cols if col in trainable.columns]
candidate_feature_cols = [col for col in candidate_feature_cols if col in trainable.columns]

final_dataset = trainable[final_cols].copy()

print("SPF feature columns:", len(spf_feature_cols))
print("Past-error feature columns:", len(past_error_feature_cols))
print("EA-MD-QD feature columns:", len(ea_feature_cols))
print("Candidate feature columns:", len(candidate_feature_cols))
display(pd.DataFrame({"candidate_feature": candidate_feature_cols}))
display(final_dataset.head())

## Validation Checks

These checks catch the main problems that would break the NN notebook:

- missing one-year SPF forecasts
- missing realized outcomes
- duplicate forecaster-round-variable rows
- incorrectly calculated target labels
- accidental non-numeric feature columns
- obvious date leakage in the past-error features

In [ ]:
key_cols = ["survey_round", "variable", "fct_source"]

assert final_dataset["rolling_1y_forecast"].notna().all()
assert final_dataset["actual_value"].notna().all()
assert final_dataset["forecast_error"].notna().all()
assert final_dataset.duplicated(key_cols).sum() == 0

calculated_error = final_dataset["actual_value"] - final_dataset["rolling_1y_forecast"]
assert np.allclose(final_dataset["forecast_error"], calculated_error)

non_numeric_features = [
    col for col in candidate_feature_cols
    if not pd.api.types.is_numeric_dtype(trainable[col])
]
print("Non-numeric candidate features:", non_numeric_features)
assert len(non_numeric_features) == 0

# The historical error summaries should use a non-missing, conservative history cutoff.
assert (final_dataset["past_error_reference_date"] if "past_error_reference_date" in final_dataset else trainable["past_error_reference_date"]).notna().all()

ea_match_share = final_dataset[ea_feature_cols].notna().any(axis=1).mean()
print(f"Rows with at least one EA-MD-QD predictor: {ea_match_share:.1%}")

print("All validation checks passed.")

## Baseline Forecast-Error Summary

Before training a neural network, it is useful to inspect the raw SPF errors.
These are in-sample descriptive statistics, not the final evaluation metric.

In [ ]:
def rmse(values):
    return math.sqrt(np.mean(np.square(values)))


baseline_summary = (
    final_dataset
    .groupby("variable")
    .agg(
        n_rows=("forecast_error", "size"),
        mean_error=("forecast_error", "mean"),
        mae=("abs_forecast_error", "mean"),
        rmse=("forecast_error", rmse),
        mean_actual=("actual_value", "mean"),
        mean_spf_forecast=("rolling_1y_forecast", "mean"),
    )
    .reset_index()
)

display(baseline_summary)

error_by_round = (
    final_dataset
    .groupby(["survey_round", "variable"], as_index=False)
    .agg(
        mean_forecast_error=("forecast_error", "mean"),
        mean_abs_forecast_error=("abs_forecast_error", "mean"),
    )
)
display(error_by_round.tail(10))

## Save the Final Dataset

The saved CSV is the dataset used by the NN notebook. It contains one row per
survey round, variable, and forecaster, with:

- the raw SPF one-year forecast
- the realized value and forecast-error target
- SPF consensus features
- past forecast-error history features
- lag-aligned EA-MD-QD euro-area macro and financial predictors

The NN notebook should still impute and scale features inside each train split,
using only training data.

In [ ]:
final_dataset.to_csv(FINAL_DATASET_PATH, index=False)

print(f"Saved final modelling dataset to: {FINAL_DATASET_PATH}")
print("Final dataset shape:", final_dataset.shape)
display(final_dataset.tail())

## Optional: Save a Copy to Google Drive

When this notebook is run in Google Colab, this cell mounts Google Drive,
creates a project folder, and writes the same CSV there. The `try` block keeps
the notebook runnable on a local machine, where `google.colab` is not available.

In [ ]:
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/SPF_NN_Final_Assignment")
DRIVE_OUTPUT_PATH = DRIVE_OUTPUT_DIR / FINAL_DATASET_PATH.name

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    final_dataset.to_csv(DRIVE_OUTPUT_PATH, index=False)
    print(f"Saved final dataset to Google Drive: {DRIVE_OUTPUT_PATH}")
except ModuleNotFoundError:
    print("Google Colab is not available here, so the Drive copy was skipped.")
    print("Run this cell in Colab to save the CSV to your Google Drive folder.")